In [ ]:
EXP_NAME = "0615_full_pipeline"
TRAINING_DATASET = "heart_disease_train.csv"
TEST_DATASET = "heart_disease_test.csv"
FEATURE_MAP = "scm/uci_scm_config.json"
SEED = 4

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
  from google.colab import userdata
  from google.colab import drive
  drive.mount('/content/drive')
  PROJECT_ROOT = userdata.get('PROJECT_ROOT')
else:
  PROJECT_ROOT = '../..'
  if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel
from notebooks.analysis_utils import mutual_info_grouped, plot_cat_feature, plot_cont_feature, plot_cf_cont_feature_comparison, plot_cf_cat_feature_comparison

from src.config import Config

sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1)

In [ ]:
test_latents = pd.read_csv(f'{PROJECT_ROOT}/{Config.DATA_DIR}/{EXP_NAME}/test_latent_space.csv')
test_counterfactuals = pd.read_csv(f'{PROJECT_ROOT}/{Config.DATA_DIR}/{EXP_NAME}/test_counterfactuals.csv')
training_latents = pd.read_csv(f'{PROJECT_ROOT}/{Config.DATA_DIR}/{EXP_NAME}/train_latent_space.csv')
training_counterfactuals = pd.read_csv(f'{PROJECT_ROOT}/{Config.DATA_DIR}/{EXP_NAME}/train_counterfactuals.csv')
test_dataset = pd.read_csv(f'{PROJECT_ROOT}/{Config.DATA_DIR}/{TEST_DATASET}')
training_dataset = pd.read_csv(f'{PROJECT_ROOT}/{Config.DATA_DIR}/{TRAINING_DATASET}')

classifiers_results = pd.read_csv(f'{PROJECT_ROOT}/{Config.RESULTS_DIR}/{EXP_NAME}/bootstrapped_evaluation_results.csv')
classifiers_results['model'] = classifiers_results['model'].replace({
  'baseline': "01 - Baseline",
  'baseline_unaware': "02 - S-Unaware Baseline",
  'ablation': "03 - Xind/corr-only Baseline",
  'fair': "11 - Fair",
  'fair_unaware': "12 - S-Unaware Fair"
})
classifiers_results.sort_values(by='model', axis=0, inplace=True)

with open(f"{PROJECT_ROOT}/configs/{FEATURE_MAP}", 'r') as f:
  feature_map = json.load(f)

In [ ]:
training_df = training_latents.merge(training_dataset, how="inner", left_on="patient_index", right_index=True)
training_df = training_df.merge(training_counterfactuals, how="inner", on="patient_index", suffixes=("", "_cf"))
training_df.drop('sample_index_cf', axis=1, inplace=True)

test_df = test_latents.merge(test_dataset, how="inner", left_on="patient_index", right_index=True)
test_df = test_df.merge(test_counterfactuals, how="inner", on="patient_index", suffixes=("", "_cf"))
test_df.drop('sample_index_cf', axis=1, inplace=True)

In [ ]:
desc_cols = [f['name'] for f in feature_map['desc']]
desc_cf_cols = [f"{f['name']}_cf" for f in feature_map['desc']]
latent_cols = [col for col in training_latents.columns if 'u_desc_' in col]
sens_col = feature_map['sens'][0]['name']
target_col = feature_map['target']['name']

In [ ]:
from scipy.stats import sem, t
def get_95_ci(data):
  '''
    Calculates the 95% confidence interval for a given set of data.

    Inputs
      data: data as a Pandas Series

    Outputs
      interval: Array of the lower and upper bounds of the confidence interval
  '''
  n = len(data)
  mean = data.mean()
  std_err = sem(data, nan_policy='omit')
  if std_err == 0:
    return (mean, mean)
  interval = t.interval(0.95, n - 1, loc=mean, scale=std_err)
  return interval

def format_avg(metric, precision=3, perc=False):
  ci = get_95_ci(metric)
  if perc:
    return f"{round(metric.mean()*100, precision)}% ({round(ci[0]*100, precision)}%-{round(ci[1]*100, precision)}%)"
  else:
    return f"{round(metric.mean(), precision)} ({round(ci[0], precision)}-{round(ci[1], precision)})"

# Classifier results

## Global results

In [ ]:
global_results = classifiers_results[classifiers_results['subgroup'] == "Global"]
global_aggregated_results = global_results.groupby('model').agg({
  "auprc":( lambda x: format_avg(x, precision=2, perc=True)),
  "brier_score":format_avg,
  "recall":( lambda x: format_avg(x, precision=2, perc=True)),
  "ppv":( lambda x: format_avg(x, precision=2, perc=True)),
  "fnr":( lambda x: format_avg(x, precision=2, perc=True)),
  "fpr":( lambda x: format_avg(x, precision=2, perc=True)),
  "cf_harm_balanced":format_avg,
  "cf_harm_pos":format_avg,
  "cf_harm_neg":format_avg,
})

print(global_aggregated_results.reset_index().to_markdown(index=False))

## Stratified results

In [ ]:
group_results = classifiers_results[classifiers_results['subgroup'] != "Global"]
aggregated_group_results = group_results.pivot_table(index='model', columns='subgroup', values=['auprc', 'recall', 'ppv', 'fpr'], aggfunc=(lambda x: format_avg(x, precision=2, perc=True)))
aggregated_group_results.columns = [
    f"{metric} - {subgroup}" for metric, subgroup in aggregated_group_results.columns
]
aggregated_group_results = aggregated_group_results.reset_index()
print(aggregated_group_results.to_markdown(index=False))

In [ ]:
aggregated_group_cf_harm = group_results.pivot_table(index='model', columns='subgroup', values=['cf_harm_balanced', 'cf_harm_pos', 'cf_harm_neg'], aggfunc=(lambda x: format_avg(x, precision=2, perc=True)))
aggregated_group_cf_harm.columns = [
    f"{metric} - {subgroup}" for metric, subgroup in aggregated_group_cf_harm.columns
]
aggregated_group_cf_harm = aggregated_group_cf_harm.reset_index()
print(aggregated_group_cf_harm.to_markdown(index=False))

# Latent space

## Mutual Information with $X_{sens}$

### Training dataset

In [ ]:
mutual_info_grouped(
  training_df,
  feature_cols= desc_cols + latent_cols,
  target_col=sens_col,
  target_label="sensitive attribute",
  iterations=100,
  n_samples=400,
  seed=SEED
)

### Test dataset

In [ ]:
mutual_info_grouped(
  test_df,
  feature_cols= desc_cols + latent_cols,
  target_col=sens_col,
  target_label="sensitive attribute",
  iterations=100,
  n_samples=100,
  seed=SEED
)

# Counterfactuals

In [ ]:
for feature in feature_map['desc']:
  col_name = feature['name']
  cf_col_name = feature['name'] + "_cf"
  group_0_mask = training_df[sens_col] == 0
  group_1_mask = training_df[sens_col] == 1
  if feature['type'] == "continuous":
    plot_cf_cont_feature_comparison(training_df, col_name, cf_col_name, sens_col, target_col)
  else:
    plot_cf_cat_feature_comparison(training_df, col_name, cf_col_name, sens_col, target_col)